In [1]:
import random

import chemprop as cp
import lightning as L
import numpy as np
import pandas as pd
import rdkit.Chem as Chem
import torch
import wandb
from delta_data import RandomPairDataModule
from delta_model import DeltaProp, Encoder, Interaction
from lightning.pytorch.callbacks.early_stopping import EarlyStopping
from lightning.pytorch.callbacks.model_checkpoint import ModelCheckpoint
from pytorch_lightning.utilities import move_data_to_device
from ray import train, tune
from ray.tune.integration.pytorch_lightning import TuneReportCheckpointCallback
from ray.tune.schedulers import ASHAScheduler
from ray.tune.search import ConcurrencyLimiter
from ray.tune.search.optuna import OptunaSearch
from rdkit.Chem.Descriptors import CalcMolDescriptors
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.Scaffolds.MurckoScaffold import MurckoScaffoldSmiles
from rdkit.rdBase import BlockLogs
from sklearn.model_selection import GroupShuffleSplit

wandb.login(key="cf344975eb80edf6f0d52af80528cc6094234caf")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: rahul-e-dev to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [2]:
RANDOM_SEED = 42


def set_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seeds(RANDOM_SEED)

In [3]:
def standardize(smiles):
    with BlockLogs():
        params = Chem.SmilesParserParams()
        params.removeHs = False
        mol = Chem.MolFromSmiles(smiles, params)  # type: ignore
        if mol is None:
            return None

        clean_mol = rdMolStandardize.Cleanup(mol)
        parent_clean_mol = rdMolStandardize.FragmentParent(clean_mol)
        uncharger = rdMolStandardize.Uncharger()
        uncharged_parent_clean_mol = uncharger.uncharge(parent_clean_mol)
        return uncharged_parent_clean_mol


def generate_features(df):
    with BlockLogs():
        feats = pd.DataFrame.from_records(df["mol"].map(CalcMolDescriptors).tolist())
        feats.columns = [f"feat_{f}" for f in feats.columns]
        df = pd.concat(
            [
                df.reset_index(drop=True),
                feats,
            ],
            axis=1,
        )

    return df


def mol_to_inchi(mol):
    with BlockLogs():
        return Chem.MolToInchi(mol)


def get_scaffold(mol) -> str:
    smi = Chem.MolToSmiles(mol)
    scaffold = MurckoScaffoldSmiles(smi)
    if len(scaffold) == 0:
        scaffold = smi
    return scaffold


df = pd.read_excel("./GSK_TBA_AH_JSFedit070425.xlsx")
df["target"] = df["parent_remaining_24"] / 100 + df["metabolite_detected"]
df = df.loc[:, ["smiles", "target"]]
df["mol"] = df["smiles"].map(standardize)
df = df.dropna(subset=["mol"])
df["inchi"] = df["mol"].map(mol_to_inchi)
df = df.groupby(["inchi"]).filter(lambda x: len(x) == 1).reset_index(drop=True)


df = generate_features(df)

clusters, _ = pd.factorize(df["mol"].map(get_scaffold))
clusters = pd.Series(clusters)

df = df.drop(["smiles", "inchi"], axis=1)

splitter = GroupShuffleSplit(n_splits=1, random_state=RANDOM_SEED)
train_idxs, val_test_idxs = next(splitter.split(df, groups=clusters))
df_train = df.loc[train_idxs].reset_index(drop=True)
df_val_test = df.loc[val_test_idxs].reset_index(drop=True)
clusters_val_test = clusters.iloc[val_test_idxs].reset_index(drop=True)

splitter = GroupShuffleSplit(n_splits=1, random_state=RANDOM_SEED, test_size=0.5)
val_idxs, test_idxs = next(splitter.split(df_val_test, groups=clusters_val_test))
df_val = df_val_test.loc[val_idxs].reset_index(drop=True)
df_test = df_val_test.loc[test_idxs].reset_index(drop=True)

In [4]:
def get_molecule_datapoint(row):
    feat_entry_names = [f for f in row.index if f.startswith("feat")]
    feat_array = pd.to_numeric(row[feat_entry_names], errors="coerce")
    return cp.data.MoleculeDatapoint(
        mol=row["mol"], y=np.array([row["target"]]), x_d=feat_array.to_numpy()
    )


featurizer = cp.featurizers.SimpleMoleculeMolGraphFeaturizer()
train_mol_dataset = cp.data.MoleculeDataset(
    df_train.apply(get_molecule_datapoint, axis=1), featurizer=featurizer
)
val_mol_dataset = cp.data.MoleculeDataset(
    df_val.apply(get_molecule_datapoint, axis=1), featurizer=featurizer
)
test_mol_dataset = cp.data.MoleculeDataset(
    df_test.apply(get_molecule_datapoint, axis=1), featurizer=featurizer
)

x_d_scaler = train_mol_dataset.normalize_inputs("X_d")
val_mol_dataset.normalize_inputs("X_d", x_d_scaler)
test_mol_dataset.normalize_inputs("X_d", x_d_scaler)

train_mol_dataset.cache = True
val_mol_dataset.cache = True
test_mol_dataset.cache = True

In [5]:
def tune_func(config, train_mol_dataset, val_mol_dataset):
    torch.set_float32_matmul_precision("high")

    depth = config["depth"]
    ffn_hidden_dim = config["ffn_hidden_dim"]
    ffn_num_layers = config["ffn_num_layers"]
    message_hidden_dim = config["message_hidden_dim"]
    batch_norm = config["batch_norm"]
    encoder_dropout = config["encoder_dropout"]
    interaction_dropout = config["interaction_dropout"]

    ###############################################################################################

    mp = cp.nn.BondMessagePassing(d_h=message_hidden_dim, depth=depth)
    agg = cp.nn.NormAggregation()
    ffn_dims = mp.output_dim + train_mol_dataset.X_d.shape[-1]
    encoder = Encoder(
        input_dim=ffn_dims,
        hidden_dim=ffn_hidden_dim,
        n_layers=ffn_num_layers,
        activation=torch.nn.ELU(),
        dropout=encoder_dropout,
    )
    interaction = Interaction(encoder.output_dim, dropout=interaction_dropout)
    model = DeltaProp(mp, agg, encoder, interaction, batch_norm=batch_norm)

    ################################################################################################
    trainer = L.Trainer(
        logger=None,
        enable_checkpointing=True,
        enable_progress_bar=False,
        accelerator="auto",
        devices=1,
        max_epochs=20,
        accumulate_grad_batches=2,
        callbacks=[
            EarlyStopping(monitor="val_loss", mode="min", verbose=False, patience=10),
            TuneReportCheckpointCallback(),
        ],
    )

    trainer.fit(
        model,
        datamodule=RandomPairDataModule(
            train_mol_dataset, val_mol_dataset, 1.5, batch_size=16
        ),
    )

In [6]:
search_space = {
    "depth": tune.qrandint(lower=2, upper=6, q=1),
    "ffn_hidden_dim": tune.qrandint(lower=300, upper=2400, q=100),
    "ffn_num_layers": tune.qrandint(lower=1, upper=3, q=1),
    "message_hidden_dim": tune.qrandint(lower=300, upper=2400, q=100),
    "encoder_dropout": tune.uniform(lower=0.0, upper=0.3),
    "interaction_dropout": tune.uniform(lower=0.0, upper=0.3),
    "batch_norm": tune.choice([True, False])
}
search_alg = ConcurrencyLimiter(OptunaSearch(seed=42), max_concurrent=2)
scheduler = ASHAScheduler(max_t=20, grace_period=5, reduction_factor=2)

tune_fn = tune.with_resources(
    tune.with_parameters(
        tune_func, train_mol_dataset=train_mol_dataset, val_mol_dataset=val_mol_dataset
    ),
    resources={"GPU": 0.25},
)

# Checkpoint config controls the checkpointing behavior of Ray
checkpoint_config = tune.CheckpointConfig(
    num_to_keep=1,  # number of checkpoints to keep
    checkpoint_score_attribute="val_loss",  # Save the checkpoint based on this metric
    checkpoint_score_order="min",  # Save the checkpoint with the lowest metric value
)

tuner = tune.Tuner(
    tune_fn,
    param_space=search_space,
    tune_config=tune.TuneConfig(
        metric="val_loss",
        mode="min",
        num_samples=20,
        scheduler=scheduler,
        search_alg=search_alg,
    ),
    run_config=tune.RunConfig(
        checkpoint_config=tune.CheckpointConfig(
            num_to_keep=1,
            checkpoint_score_attribute="val_loss",
            checkpoint_score_order="min",
        ),
        failure_config=train.FailureConfig(max_failures=3),
    ),
)

results = tuner.fit()
_, best_result = results.get_best_result().best_checkpoints[0]
best_config = best_result["config"]

(tune_func pid=63048) 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
(tune_func pid=63048) GPU available: True (cuda), used: True
(tune_func pid=63048) TPU available: False, using: 0 TPU cores
(tune_func pid=63048) HPU available: False, using: 0 HPUs
(tune_func pid=63048) LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
(tune_func pid=63048) Loading `train_dataloader` to estimate number of stepping batches.
(tune_func pid=63048) /root/delta/.venv/lib/python3.12/site-packages/lightning/pytorch/loops/fit_loop.py:310: The number of training batches (25) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.
(tune_func pid=63048) 
(tune_func pid=63048)   | Name            | Type               | Params | Mode 
(tune_func pid=63048) -----------

In [7]:
@torch.no_grad()
def embed_all(mol_dataset: cp.data.datasets.MoleculeDataset, model):
    dl = torch.utils.data.DataLoader(
        mol_dataset,
        batch_size=64,
        shuffle=False,
        collate_fn=cp.data.dataloader.collate_batch,
    )
    all_embeds = []
    for batch in dl:
        batch = move_data_to_device(batch, model.device)
        res = model.embed_simple_batch(batch)
        all_embeds.append(res["embeds"])

    all_embeds = torch.cat(all_embeds)
    return all_embeds

In [9]:
from lightning.pytorch.loggers import WandbLogger
torch.set_float32_matmul_precision('high')

depth = best_config["depth"]
ffn_hidden_dim = best_config["ffn_hidden_dim"]
ffn_num_layers = best_config["ffn_num_layers"]
message_hidden_dim = best_config["message_hidden_dim"]
batch_norm = best_config['batch_norm']
encoder_dropout = best_config["encoder_dropout"]
interaction_dropout = best_config["interaction_dropout"]

###############################################################################################

mp = cp.nn.BondMessagePassing(d_h=message_hidden_dim, depth=depth)
agg = cp.nn.NormAggregation()
ffn_dims = mp.output_dim + train_mol_dataset.X_d.shape[-1]
encoder = Encoder(
    input_dim=ffn_dims, 
    hidden_dim=ffn_hidden_dim, 
    n_layers=ffn_num_layers, 
    activation=torch.nn.ELU(), 
    dropout=encoder_dropout
)
interaction = Interaction(encoder.output_dim, dropout=interaction_dropout)
model = DeltaProp(mp, agg, encoder, interaction, batch_norm=batch_norm)


################################################################################################
wandb.finish()
wandb_logger = WandbLogger(project="deltaprop_tba", log_model="all", save_code=True)
wandb_logger.watch(model, log="gradients", log_freq=50) 
wandb_logger.experiment.mark_preempting()

trainer = L.Trainer(
    logger=wandb_logger,
    enable_checkpointing=True,
    enable_progress_bar=True,
    accelerator="auto",
    devices=1,
    max_epochs=20,
    callbacks=[
        EarlyStopping(monitor="val_loss", mode="min", verbose=True, patience=10),
        ModelCheckpoint(monitor="val_loss", mode="min", save_top_k=1)
    ],
)

trainer.fit(model, datamodule=RandomPairDataModule(train_mol_dataset, val_mol_dataset, 1.5, batch_size=8))
model = DeltaProp.load_from_checkpoint(trainer.checkpoint_callback.best_model_path)

##################################################################################################

train_embeds = embed_all(train_mol_dataset, model)
test_embeds = embed_all(test_mol_dataset, model)

exemplar_idxs = np.argwhere(train_mol_dataset.Y.squeeze() > 1.5)
exemplar_embeds = train_embeds[exemplar_idxs].squeeze()

with torch.no_grad():
    pred_probs = model.interaction(test_embeds, exemplar_embeds).sigmoid().mean(axis=-1)
    preds = (pred_probs >= 0.5).float()

    pred_probs = pred_probs.cpu().detach().numpy().squeeze()
    preds = preds.cpu().detach().numpy().squeeze()
    labels = df_test['target'] > 1.5

wandb: logging graph, to disable use `wandb.watch(log_graph=False)`
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.

  | Name            | Type               | Params | Mode 
---------------------------------------------------------------
0 | message_passing | BondMessagePassing | 2.2 M  | train
1 | agg             | NormAggregation    | 0      | train
2 | encoder         | Encoder            | 1.6 M  | train
3 | interaction     | Interaction        | 90.0 K | train
4 | bn              | Identity           | 0      | train
5 | X_d_transform   | Identity           | 0      | train
6 | loss_fn         | BCEWithLogitsLoss  | 0      | train
---------------------------------------------------------------
3.8 M     Trainable params
0         Non-trainable params
3.8 M     Total params
15.210    Total estimated model p

Sanity Checking: |                                                                                            …

Training: |                                                                                                   …

Validation: |                                                                                                 …

Metric val_loss improved. New best score: 0.499


...

Validation: |                                                                                                 …

Validation: |                                                                                                 …

Validation: |                                                                                                 …

Validation: |                                                                                                 …

Monitored metric val_loss did not improve in the last 10 records. Best score: 0.499. Signaling Trainer to stop.


In [10]:
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

wandb_logger.log_table(
    'final_metrics', 
    ['accuracy', 'balanced_accuracy', 'f1', 'precision', 'recall', 'AUCROC', 'PRAUC'],
    [[
        accuracy_score(labels, preds),
        balanced_accuracy_score(labels, preds),
        f1_score(labels, preds),
        precision_score(labels, preds),
        recall_score(labels, preds),
        roc_auc_score(labels, pred_probs),
        average_precision_score(labels, pred_probs)
    ]]
)

wandb.finish()

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


epoch,▁▁▁▂▂▂▂▂▂▃▃▃▄▄▄▅▅▅▅▅▅▆▆▆▇▇▇▇▇▇███
train_exemplar_sym_loss_epoch,▁▂█▆▄▃▂▁▁▂▁
train_exemplar_sym_loss_step,▁▃▄█▄▂▁▁▄▂▁
train_loss_epoch,▅▅█▇▅▃▃▂▂▂▁
train_loss_step,▆▇█▇▄▆▇▄▅▁▄
train_lr_exemplar_loss_epoch,█▆█▇▅▃▄▂▂▂▁
train_lr_exemplar_loss_step,▆▇█▆▃▇█▅▅▁▅
train_rl_exemplar_loss_epoch,▇▆█▇▅▃▄▂▂▂▁
train_rl_exemplar_loss_step,▆▇█▆▄▇█▅▅▁▅
trainer/global_step,▁▁▁▂▂▂▂▂▂▃▃▃▄▄▄▅▅▅▅▅▅▆▆▆▇▇▇▇▇▇████
val_exemplar_sym_loss,▂▅█▇▃▂▂▁▁▁▁
